In [3]:
import os
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
# from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score
)
import shap


DATA_DIR = os.environ.get("DATA_DIR", "../data")
conn = duckdb.connect(f"{DATA_DIR}/warehouse.duckdb", read_only=True)

In [ ]:
lines = conn.sql("""
    SELECT
        po_number,
        item_id,
        category,
        unit_of_measure,
        order_date,
        vendor_id,
        quantity_ordered,
        unit_price_paid,
        status
    FROM gold.fct_company_a_purchase_line
    WHERE status <> 'Cancelled'
""").df()

lines["order_date"] = pd.to_datetime(lines["order_date"])
lines["extended"] = lines["quantity_ordered"] * lines["unit_price_paid"]

n_orders = lines["po_number"].nunique()
window = f"{lines['order_date'].min():%b %Y} – {lines['order_date'].max():%b %Y}"

print(lines["status"].value_counts())
print(f"{n_orders} orders, {lines['item_id'].nunique()} items, {window}")

In [ ]:
# One row per item, so "how often" and "how much money" can be ranked separately.
# An item counts once per order, even when that order has two lines for it.
# Eligible items were bought in at least 8 different months and across a full year.
# Anything shorter cannot answer "is there a better time of year".

items = (
    lines.groupby(["item_id", "category", "unit_of_measure"], as_index=False)
    .agg(
        n_orders=("po_number", "nunique"),
        n_lines=("po_number", "size"),
        n_months=("order_date", lambda s: s.dt.to_period("M").nunique()),
        first_order=("order_date", "min"),
        last_order=("order_date", "max"),
        n_vendors=("vendor_id", "nunique"),
        median_qty=("quantity_ordered", "median"),
        total_spend=("extended", "sum"),
        n_priced=("unit_price_paid", "count"),
    )
)

items["span_days"] = (items["last_order"] - items["first_order"]).dt.days
items["attachment"] = items["n_orders"] / n_orders
# Reorder rate over the stretch they were actually buying it.
# A three-month burst can score high here and still fail the span floor.
items["orders_per_month"] = items["n_orders"] / (items["span_days"] / 30.44).clip(lower=1)

SPAN_MONTHS = 8
SPAN_DAYS = 365
items["eligible"] = (items["n_months"] >= SPAN_MONTHS) & (items["span_days"] >= SPAN_DAYS)